In [0]:
bronze_train = spark.table(
    "imdb_sentiment.bronze.train_reviews"
)

print("Training rows:", bronze_train.count())
bronze_train.printSchema()

Training rows: 25000
root
 |-- text: string (nullable = true)
 |-- label: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

bronze_train = spark.table(
    "imdb_sentiment.bronze.train_reviews"
)

silver_train = (
    bronze_train
    .withColumn(
        "clean_text",
        F.trim(
            F.regexp_replace(
                F.col("text"),
                r"(?i)<br\s*/?>",
                " "
            )
        )
    )
    .withColumn(
        "text_length",
        F.length(F.col("clean_text"))
    )
)

display(
    silver_train.select(
        "text",
        "clean_text",
        "label",
        "text_length"
    ).limit(5)
)

text,clean_text,label,text_length
"the author of the book, by the same title, should not have let her name be used for this movie. if you have read the book, this movie takes such a liberal interpretation of the actual events in the book and its spirit that the movie and book seem to have quite little in common except the title and some superficial details. the movie adds nothing, in terms of artistic merit, to the book's own literary achievement.for those who have not read the book: you will also be disappointed. not only does the plot move at an incredibly slow pace, it doesn't offer anything more while it is moving slowly (like character development, for example). some viewers might be entertained by some of the graphic lesbian love scenes later on in the movie, but you might as well watch a showtime special for the stuff they show in therese and isabelle--its fairly tame and not imaginative at all.","the author of the book, by the same title, should not have let her name be used for this movie. if you have read the book, this movie takes such a liberal interpretation of the actual events in the book and its spirit that the movie and book seem to have quite little in common except the title and some superficial details. the movie adds nothing, in terms of artistic merit, to the book's own literary achievement. for those who have not read the book: you will also be disappointed. not only does the plot move at an incredibly slow pace, it doesn't offer anything more while it is moving slowly (like character development, for example). some viewers might be entertained by some of the graphic lesbian love scenes later on in the movie, but you might as well watch a showtime special for the stuff they show in therese and isabelle--its fairly tame and not imaginative at all.",0,882
"I'm not sure how related they are, but I'm almost certain that Lost and Delirious is a remake of this movie (or the story that it's based on). Very similar plotline, and even some of the scenes and sets seem to be very, very similar. Lost & Delirious is actually a much better movie, so see that one instead.This one moves very slowly, but being a late 60s French movie, that is to be expected of the style. Told in a retrospect from the perspective of one of the girls revisiting the school. The editing of the flashbacks with the current scenes is a little bit confusing at first, particularly since the audio from each overlaps (ie, hearing flashbacks while seeing the present and vice versa). Also, the ""girls"" are a bit old to think that they are in a boarding school. Finally, not much character development to even get you attached to the movie.","I'm not sure how related they are, but I'm almost certain that Lost and Delirious is a remake of this movie (or the story that it's based on). Very similar plotline, and even some of the scenes and sets seem to be very, very similar. Lost & Delirious is actually a much better movie, so see that one instead. This one moves very slowly, but being a late 60s French movie, that is to be expected of the style. Told in a retrospect from the perspective of one of the girls revisiting the school. The editing of the flashbacks with the current scenes is a little bit confusing at first, particularly since the audio from each overlaps (ie, hearing flashbacks while seeing the present and vice versa). Also, the ""girls"" are a bit old to think that they are in a boarding school. Finally, not much character development to even get you attached to the movie.",0,854
"Shallow, shallow script ...stilted acting ...the shadows of boom mikes lingering over the actors' heads in scenes ...worth watching because Kate Mulgrew plays the most selfish mother in TV movie history and it's all before Ben Affleck got his teeth capped.","Shallow, shallow script ...stilted acting ...the shadows of boom mikes lingering over the actors' heads in scenes ...worth watching because Kate Mulgrew plays the most selfish mother in TV movie history and it's 

In [0]:
from pyspark.sql import functions as F

silver_train.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("clean_text").isNull().cast("int")).alias("null_reviews"),
    F.sum((F.length("clean_text") == 0).cast("int")).alias("empty_reviews"),
    F.sum(F.col("label").isin(0, 1).eqNullSafe(False).cast("int")).alias("invalid_labels"),
    F.sum(F.col("clean_text").rlike(r"(?i)<br\s*/?>").cast("int")).alias("remaining_br_tags")
).show()

+----------+------------+-------------+--------------+-----------------+
|total_rows|null_reviews|empty_reviews|invalid_labels|remaining_br_tags|
+----------+------------+-------------+--------------+-----------------+
|     25000|           0|            0|             0|                0|
+----------+------------+-------------+--------------+-----------------+



In [0]:
silver_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("imdb_sentiment.silver.train_reviews")

print("Silver training table saved.")

print(
    "Rows saved:",
    spark.table("imdb_sentiment.silver.train_reviews").count()
)

Silver training table saved.
Rows saved: 25000


In [0]:
from pyspark.sql import functions as F

bronze_test = spark.table(
    "imdb_sentiment.bronze.test_reviews"
)

silver_test = (
    bronze_test
    .withColumn(
        "clean_text",
        F.trim(
            F.regexp_replace(
                F.col("text"),
                r"(?i)<br\s*/?>",
                " "
            )
        )
    )
    .withColumn(
        "text_length",
        F.length(F.col("clean_text"))
    )
)

print("Test rows:", silver_test.count())

display(
    silver_test.select(
        "text",
        "clean_text",
        "label",
        "text_length"
    ).limit(5)
)

Test rows: 25000


text clean_text label text_length In the world of "shorts" (most of which aren't), this film is a gem. A quiet, concise peek into the world of a young woman who's a reader for a blind woman, here the stellar Elizabeth Franz - this film bears the textures, layers and visual storytelling of a sumptuously painted still life. The dialogue is minimal, the cinematography is stunning, and the direction sure, clear and compelling. I saw this film in a film festival held in a loud and crowded Tribeca bar - and within the first two minutes (and for the first time that night), the crowd fell quiet. That says it all. In the world of "shorts" (most of which aren't), this film is a gem. A quiet, concise peek into the world of a young woman who's a reader for a blind woman, here the stellar Elizabeth Franz - this film bears the textures, layers and visual storytelling of a sumptuously painted still life. The dialogue is minimal, the cinematography is stunning, and the direction sure, clear and compelling. I saw this film in a film festival held in a loud and crowded Tribeca bar - and within the first two minutes (and for the first time that night), the crowd fell quiet. That says it all. 1 581 Saw this last night at the 7th Annual NYC Home Film Festival -- a rather tumultuous venue that quieted down for this sweet and touching two-woman film. The relationship between the two -- one, older and blind, the other, a twenty- something who helps her with her mail -- is established deftly, and its depth is readily apparent. There are no car chases, explosions, or mind-boggling special effects -- just the sometimes difficult but rewarding task of humans reaching out toward each other to help and be helped. See if it you can! Saw this last night at the 7th Annual NYC Home Film Festival -- a rather tumultuous venue that quieted down for this sweet and touching two-woman film. The relationship between the two -- one, older and blind, the other, a twenty- something who helps her with her mail -- is established deftly, and its depth is readily apparent. There are no car chases, explosions, or mind-boggling special effects -- just the sometimes difficult but rewarding task of humans reaching out toward each other to help and be helped. See if it you can! 1 536 I saw this film on the same night I saw 6 other shorts. This one was leaps and bounds ahead of the others in terms of quality of acting, directing, filming and originality of story. It comes together with a professionalism often lacking in short films. This is a great short film for the following reasons: 1. Amazing performances. The 2 actresses are both compelling and believable instantly. Their chemistry is palpable; the depth of their relationship is conveyed, even though the film is 10 mins long. The acting and directing are very powerful. 2. Beautifully shot. The lighting, framing and general filming are striking. The 35mm film sets this movie apart from other shorts. And what a difference it makes. The filming is worthy of this film stock. 3. Strong story. Often short films try to do too little or too much with the time. Or they are trying to tell a predictable story. This is a good balance of telling you an interesting story, without drawing it out too long or leaving you unsatisfied. You wonder what will happen next, while at the same time, feeling you saw something happen. Nice score, too. Compliments without overpowering. If you can find this at a festival, make an effort to see it. I saw this film on the same night I saw 6 other shorts. This one was leaps and bounds ahead of the others in terms of quality of acting, directing, filming and originality of story. It comes together with a professionalism often lacking in short films. This is a great short film for the following reasons: 1. Amazing performances. The 2 actresses are both compelling and believable instantly. Their chemistry is palpable; the depth of their relationship is conveyed, even though the film is 10 mins long. The acting

In [0]:
from pyspark.sql import functions as F

silver_test.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("clean_text").isNull().cast("int")).alias("null_reviews"),
    F.sum((F.length("clean_text") == 0).cast("int")).alias("empty_reviews"),
    F.sum((~F.col("label").isin(0, 1)).cast("int")).alias("invalid_labels"),
    F.sum(F.col("clean_text").rlike(r"(?i)<br\s*/?>").cast("int")).alias("remaining_br_tags")
).show()

+----------+------------+-------------+--------------+-----------------+
|total_rows|null_reviews|empty_reviews|invalid_labels|remaining_br_tags|
+----------+------------+-------------+--------------+-----------------+
|     25000|           0|            0|             0|                0|
+----------+------------+-------------+--------------+-----------------+



In [0]:
silver_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("imdb_sentiment.silver.test_reviews")

print("Silver test table saved.")

print(
    "Rows saved:",
    spark.table("imdb_sentiment.silver.test_reviews").count()
)

Silver test table saved.
Rows saved: 25000


In [0]:
from pyspark.sql import functions as F

train = spark.table("imdb_sentiment.silver.train_reviews")
test = spark.table("imdb_sentiment.silver.test_reviews")

# Count repeated review texts within each split
train_duplicates = (
    train.groupBy("clean_text")
    .count()
    .filter(F.col("count") > 1)
)

test_duplicates = (
    test.groupBy("clean_text")
    .count()
    .filter(F.col("count") > 1)
)

# Find review texts appearing in both splits
overlap = (
    train.select("clean_text").distinct()
    .join(
        test.select("clean_text").distinct(),
        on="clean_text",
        how="inner"
    )
)

print("Training duplicate text groups:", train_duplicates.count())
print("Test duplicate text groups:", test_duplicates.count())
print("Unique review texts shared between splits:", overlap.count())

Training duplicate text groups: 93
Test duplicate text groups: 191
Unique review texts shared between splits: 123


In [0]:
from pyspark.sql import functions as F

train = spark.table("imdb_sentiment.silver.train_reviews")
test = spark.table("imdb_sentiment.silver.test_reviews")

train_texts = train.select(
    "clean_text",
    F.col("label").alias("train_label")
).distinct()

test_texts = test.select(
    "clean_text",
    F.col("label").alias("test_label")
).distinct()

overlap_details = (
    train_texts
    .join(test_texts, on="clean_text", how="inner")
)

print("Shared unique review texts:", overlap_details.count())

print("Label agreement:")
display(
    overlap_details
    .groupBy("train_label", "test_label")
    .count()
    .orderBy("train_label", "test_label")
)

Shared unique review texts: 123
Label agreement:


train_label,test_label,count
0,0,95
1,1,28


In [0]:
from pyspark.sql import functions as F

train = spark.table("imdb_sentiment.silver.train_reviews")
test = spark.table("imdb_sentiment.silver.test_reviews")

# Identify unique review texts shared by both datasets
overlap_texts = (
    train.select("clean_text").distinct()
    .join(
        test.select("clean_text").distinct(),
        on="clean_text",
        how="inner"
    )
)

# Remove overlapping texts from training
train_deduplicated = train.join(
    overlap_texts,
    on="clean_text",
    how="left_anti"
)

print("Original training rows:", train.count())
print("Overlapping unique texts:", overlap_texts.count())
print("Training rows after leakage removal:", train_deduplicated.count())
print("Test rows retained:", test.count())

Original training rows: 25000
Overlapping unique texts: 123
Training rows after leakage removal: 24877
Test rows retained: 25000


In [0]:
# Check training class balance after leakage removal

from pyspark.sql import functions as F

display(
    train_deduplicated
    .groupBy("label")
    .agg(
        F.count("*").alias("review_count"),
        F.round(
            F.count("*") * 100.0 / train_deduplicated.count(),
            2
        ).alias("percentage")
    )
    .orderBy("label")
)

label,review_count,percentage
0,12405,49.87
1,12472,50.13
